# Delhi AQI data preparation

This notebook combines hourly city-level AQI from CPCB workbooks (2018–2025) with pollutant and weather readings averaged across six Delhi stations. Original AQI values are retained. A review-status column flags the 2021 months that need special handling in Notebook 02. Run from the `notebooks/` folder.


## 1. Choose the AQI target source and mark the 2021 review period

The primary AQI source remains the hourly city-level CPCB workbooks for all years. A comparison using AQI derived from the six-station PM data reports strong agreement in October–December 2021, but little agreement in January–September. At 4 PM on 20 August, the PM-derived estimate is 119 and the CPCB bulletin is 111, while the workbook reads 256. The December 20 workbook value (332) matches its CPCB bulletin.

Therefore the source workbook is preserved unchanged, but January–September 2021 AQI is marked `review_required` and excluded from the EDA calculations. October–December 2021 is marked `crosscheck_consistent`. This keeps winter 2021–22 available (November–December 2021 plus January–February 2022) while the winter containing January–February 2021 is excluded from the primary winter analysis. The PM-derived comparison is a proxy, not a replacement official target; no repaired values are inserted.


In [1]:
import glob
import pandas as pd


## 2. Build the hourly AQI series for 2018–2025

Read the hourly city-level workbooks with the same steps for every year. The review-status field records the special 2021 source check without changing the original AQI values.


In [2]:
months = {1:'January',2:'February',3:'March',4:'April',5:'May',6:'June',7:'July',8:'August',9:'September',10:'October',11:'November',12:'December'}
years = range(2018, 2026)
all_data = []
for year in years:
    for num, name in months.items():
        file = glob.glob(f"../aqi_target/{year}/AQI_Data_City_Level_Delhi_{name}_{year}_Hourly.xlsx")
        if not file:
            file = glob.glob(f"../aqi_target/{year}/AQI_Data_City_Level_Delhi_{name}_{year}_Hourly*.xlsx")
        df = pd.read_excel(file[0])
        long = df.melt(id_vars="Date", var_name="Hour", value_name="AQI")
        hour_clean = long["Hour"].astype(str).str[:5]
        long["Timestamp"] = pd.to_datetime(
            f"{year}-{num:02d}-" + long["Date"].astype(str).str.zfill(2) + " " + hour_clean,
            format="%Y-%m-%d %H:%M"
        )
        long = long.drop(columns=["Date", "Hour"])
        long["AQI_source"] = "official_city_level"
        long["AQI_review_status"] = "not_reviewed"
        if year == 2021:
            long.loc[long["Timestamp"].dt.month.between(1, 9), "AQI_review_status"] = "review_required"
            long.loc[long["Timestamp"].dt.month.between(10, 12), "AQI_review_status"] = "crosscheck_consistent"
        all_data.append(long)

aqi_all_years = pd.concat(all_data).sort_values("Timestamp").reset_index(drop=True)
aqi_all_years_idx = aqi_all_years.set_index("Timestamp")
aqi_all_years.shape


(70128, 4)

## 3. Load pollutant and weather data from six stations

In [3]:
stations = ["anand_vihar","rohini","rk_puram","punjabi_bagh","ito","okhla"]
station_dfs = []
for station in stations:
    files = glob.glob(f"../raw_data/{station}/*.csv")
    dfs = [pd.read_csv(f) for f in files]
    for d in dfs:
        d['Timestamp'] = pd.to_datetime(d['Timestamp'], format='mixed', dayfirst=True)
    sdf = pd.concat(dfs).sort_values('Timestamp').reset_index(drop=True)
    sdf = sdf[sdf['Timestamp'].dt.year.between(2018, 2025)]
    station_dfs.append(sdf)
all_stations = pd.concat(station_dfs).reset_index(drop=True)
all_stations.shape

(420768, 25)

## 4. Average station readings and combine them with AQI

In [4]:
numeric_cols = all_stations.select_dtypes(include='number').columns
city_avg = all_stations.groupby('Timestamp')[numeric_cols].mean().reset_index()
city_avg.shape

(70128, 25)

In [5]:
final = pd.merge(city_avg.set_index('Timestamp'), aqi_all_years_idx, left_index=True, right_index=True, how='inner')
final.shape
final.head()

,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),...,WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s),AQI,AQI_source,AQI_review_status
Timestamp,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:00:00,557.5000,684.000000,285.650000,106.236667,NaN,101.530000,18.620000,7.036667,32.846667,11.790000,...,0.300000,149.223333,NaN,0.0,8.110000,731.00,-0.045,413.0,official_city_level,not_reviewed
2018-01-01 01:00:00,556.5000,659.833333,389.325000,88.193333,NaN,102.106667,8.446667,7.366667,32.133333,10.906667,...,0.300000,127.223333,NaN,0.0,8.193333,730.96,-0.040,414.0,official_city_level,not_reviewed
2018-01-01 02:00:00,553.4375,622.666667,334.100000,74.326667,NaN,104.786667,13.540000,6.240000,30.170000,10.646667,...,0.303333,110.110000,NaN,0.0,8.166667,730.96,-0.040,415.0,official_city_level,not_reviewed
2018-01-01 03:00:00,530.9175,590.333333,208.696667,64.690000,369.60,95.913333,8.285000,3.636667,25.350000,10.436667,...,0.350000,113.250000,NaN,0.0,8.333333,730.71,-0.045,415.0,official_city_level,not_reviewed
2018-01-01 04:00:00,420.2500,568.250000,161.626667,76.763333,300.97,89.293333,5.836667,2.293333,20.103333,9.256667,...,0.410000,139.026667,NaN,0.0,8.833333,731.21,-0.010,415.0,official_city_level,not_reviewed


## 5. Save the processed dataset

In [6]:
final.to_csv("../processed_data/delhi_6station_avg_2018_2025.csv")